In [ ]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .master('local[*]')
    .appName('adi-dev')
    .getOrCreate()
)

spark.sparkContext.setLogLevel('ERROR')

In [ ]:
from pyspark.sql import functions as F

from adi.io import CsvStore, TrialBalanceRepository
from adi.pipeline import TrialBalancePipeline
from adi.config.settings import TABLE_PATHS
from adi.enrichments import (
    TransformationManager, 
    ReferenceManager
)

from finmap import FinMapClient

In [ ]:
store = CsvStore(spark, table_paths=TABLE_PATHS)
repository = TrialBalanceRepository(store)

transformation_manager = TransformationManager(spark)
reference_manager = ReferenceManager(spark)

finmap = FinMapClient.from_csv(
    spark=spark,
    metadata_path='data/reference/mapping_meta.csv',
    data_path='data/reference/mapping_data.csv',
)

pipeline = TrialBalancePipeline(
    transformation_manager=transformation_manager,
    reference_manager=reference_manager,
    finmap=finmap,
)

In [ ]:
df_source = repository.read_source()

df_source.show()

In [ ]:
df_source = repository.read_source()

df_staging = pipeline.staging(df_source)
repository.write_staging(df_staging)

df_staging_reloaded = repository.read_staging()

display(df_staging_reloaded.toPandas())

In [ ]:
(
    df_staging_reloaded
    .select('SRC_RECORD_ID', 'SRC_CLIENT_ID', 'CPTY_REF_ID', 'CLIENT_ID_TYPE', 'INTERGROUP_IND')
    .filter(F.col('SRC_RECORD_ID') == 'rec-16')
    .show()
)

In [ ]:
df_staging_reloaded.printSchema()